# License Plate Detection — YOLO

This notebook has two runnable modes:

- `TRAIN_MODE = "mini"` — creates a small local YOLO dataset for fast testing.
- `TRAIN_MODE = "full"` — trains on the full KaggleHub dataset for final results.

It also includes validation metrics and saved prediction examples.

## 1. Install and import dependencies

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path


def install_if_missing(package_name, import_name=None):
    """Install a package only if it is not already available."""
    if import_name is None:
        import_name = package_name

    if importlib.util.find_spec(import_name) is None:
        print(f"{package_name} not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
    else:
        print(f"{package_name} is already installed.")


install_if_missing("ultralytics")
install_if_missing("kagglehub")

import random
import shutil
import yaml
import pandas as pd
import torch
import kagglehub
from ultralytics import YOLO
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd()
print("Project root:", PROJECT_ROOT)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. Mini training should still run; full training will be slow on CPU.")

## 2. Download / locate the Kaggle dataset

KaggleHub uses a local cache, so it should not redownload the dataset every time if it is already available.

In [ ]:
DATASET_NAME = "barkataliarbab/license-plate-detection-dataset-10125-images"

dataset_path = Path(kagglehub.dataset_download(DATASET_NAME))
print("Dataset path:", dataset_path)
print("Top-level files/folders:")
for p in list(dataset_path.iterdir())[:20]:
    print(" -", p.name)

## 3. Find image and label folders

The original dataset paths can be slightly awkward, so this cell finds the folders automatically.

In [ ]:
def find_dir(root, target_suffix):
    """Find a directory inside root whose normalized path ends with target_suffix."""
    root = Path(root)
    matches = [
        p for p in root.rglob("*")
        if p.is_dir() and str(p).replace("\\", "/").endswith(target_suffix)
    ]

    if not matches:
        raise FileNotFoundError(f"Could not find directory ending with: {target_suffix}")

    return matches[0]


def labels_dir_from_images_dir(images_dir):
    """Convert .../images to .../labels safely."""
    images_dir = Path(images_dir)
    if images_dir.name != "images":
        raise ValueError(f"Expected image folder to end with 'images', got: {images_dir}")
    return images_dir.parent / "labels"


train_images_dir = find_dir(dataset_path, "train/images")
valid_images_dir = find_dir(dataset_path, "valid/images")
test_images_dir = find_dir(dataset_path, "test/images")

train_labels_dir = labels_dir_from_images_dir(train_images_dir)
valid_labels_dir = labels_dir_from_images_dir(valid_images_dir)
test_labels_dir = labels_dir_from_images_dir(test_images_dir)

for name, image_dir, label_dir in [
    ("train", train_images_dir, train_labels_dir),
    ("valid", valid_images_dir, valid_labels_dir),
    ("test", test_images_dir, test_labels_dir),
]:
    print(f"{name} images:", image_dir, "exists:", image_dir.exists())
    print(f"{name} labels:", label_dir, "exists:", label_dir.exists())
    print(f"{name} image count:", len(list(image_dir.glob('*'))))
    print(f"{name} label count:", len(list(label_dir.glob('*.txt'))))
    print()

## 4. Create a YOLO YAML for the full dataset

Use this for final full training.

In [ ]:
FULL_DATA_YAML = PROJECT_ROOT / "data_full.yaml"

full_data_config = {
    "path": str(dataset_path.resolve()),
    "train": str(train_images_dir.relative_to(dataset_path)),
    "val": str(valid_images_dir.relative_to(dataset_path)),
    "test": str(test_images_dir.relative_to(dataset_path)),
    "nc": 1,
    "names": ["License_Plate"],
}

with open(FULL_DATA_YAML, "w") as f:
    yaml.safe_dump(full_data_config, f, sort_keys=False)

print("Created full dataset YAML:", FULL_DATA_YAML)
print(yaml.safe_dump(full_data_config, sort_keys=False))

## 5. Create a mini dataset for testing

This physically copies a small number of images and labels into `mini_license_plate_dataset/`. This folder is ignored by Git.

In [ ]:
MINI_DATASET_DIR = PROJECT_ROOT / "mini_license_plate_dataset"
MINI_DATA_YAML = MINI_DATASET_DIR / "data.yaml"

N_TRAIN = 100
N_VALID = 20
N_TEST = 20
SEED = 42


def list_image_files(images_dir):
    images_dir = Path(images_dir)
    return sorted(
        list(images_dir.glob("*.jpg")) +
        list(images_dir.glob("*.jpeg")) +
        list(images_dir.glob("*.png"))
    )


def copy_yolo_subset(images_dir, labels_dir, output_images_dir, output_labels_dir, n_images, seed=42):
    random.seed(seed)

    output_images_dir.mkdir(parents=True, exist_ok=True)
    output_labels_dir.mkdir(parents=True, exist_ok=True)

    image_files = list_image_files(images_dir)
    print(f"Found {len(image_files)} images in {images_dir}")

    random.shuffle(image_files)
    copied = 0

    for img_path in image_files:
        label_path = Path(labels_dir) / f"{img_path.stem}.txt"

        if label_path.exists():
            shutil.copy2(img_path, output_images_dir / img_path.name)
            shutil.copy2(label_path, output_labels_dir / label_path.name)
            copied += 1

        if copied >= n_images:
            break

    print(f"Copied {copied} images to {output_images_dir}")

    if copied == 0:
        raise RuntimeError("No images were copied. Check image/label folder paths.")


def create_mini_dataset(force_recreate=False):
    if MINI_DATASET_DIR.exists() and MINI_DATA_YAML.exists() and not force_recreate:
        print("Mini dataset already exists:", MINI_DATASET_DIR)
        return MINI_DATA_YAML

    if MINI_DATASET_DIR.exists():
        shutil.rmtree(MINI_DATASET_DIR)

    copy_yolo_subset(
        train_images_dir,
        train_labels_dir,
        MINI_DATASET_DIR / "train" / "images",
        MINI_DATASET_DIR / "train" / "labels",
        n_images=N_TRAIN,
        seed=SEED,
    )

    copy_yolo_subset(
        valid_images_dir,
        valid_labels_dir,
        MINI_DATASET_DIR / "valid" / "images",
        MINI_DATASET_DIR / "valid" / "labels",
        n_images=N_VALID,
        seed=SEED,
    )

    copy_yolo_subset(
        test_images_dir,
        test_labels_dir,
        MINI_DATASET_DIR / "test" / "images",
        MINI_DATASET_DIR / "test" / "labels",
        n_images=N_TEST,
        seed=SEED,
    )

    mini_data_config = {
        "path": str(MINI_DATASET_DIR.resolve()),
        "train": "train/images",
        "val": "valid/images",
        "test": "test/images",
        "nc": 1,
        "names": ["License_Plate"],
    }

    with open(MINI_DATA_YAML, "w") as f:
        yaml.safe_dump(mini_data_config, f, sort_keys=False)

    print("Created mini dataset YAML:", MINI_DATA_YAML)
    print(yaml.safe_dump(mini_data_config, sort_keys=False))

    return MINI_DATA_YAML


mini_data_yaml = create_mini_dataset(force_recreate=False)

In [ ]:
# Check mini dataset counts
for split in ["train", "valid", "test"]:
    image_dir = MINI_DATASET_DIR / split / "images"
    label_dir = MINI_DATASET_DIR / split / "labels"
    print(split)
    print("  images:", len(list_image_files(image_dir)))
    print("  labels:", len(list(label_dir.glob("*.txt"))))

## 6. Add generated files to `.gitignore`

This keeps the repo clean. The notebook can recreate these files locally.

In [ ]:
gitignore_path = PROJECT_ROOT / ".gitignore"

entries = [
    "mini_license_plate_dataset/",
    "license_plate_detection/",
    "runs/",
    "*.pt",
    "*.onnx",
    "data_full.yaml",
    ".ipynb_checkpoints/",
    "__pycache__/",
]

existing = set()
if gitignore_path.exists():
    existing = set(gitignore_path.read_text().splitlines())

with open(gitignore_path, "a") as f:
    for entry in entries:
        if entry not in existing:
            f.write(entry + "\n")

print(".gitignore updated:", gitignore_path)
print(gitignore_path.read_text())

## 7. Choose training mode

Use `mini` for quick testing and `full` for final training.

- Mini: 100 train / 20 valid / 20 test images, small image size, few epochs.
- Full: complete dataset, larger image size, more epochs.

In [ ]:
# Change this to "full" for final training.
TRAIN_MODE = "mini"  # "mini" or "full"

MODEL_WEIGHTS = "yolo11n.pt"
PROJECT_NAME = "license_plate_detection"
DEVICE = 0 if torch.cuda.is_available() else "cpu"

TRAINING_CONFIGS = {
    "mini": {
        "data": str(mini_data_yaml),
        "epochs": 3,
        "imgsz": 320,
        "batch": 4,
        "name": "yolo11n_mini_test",
        "patience": 5,
    },
    "full": {
        "data": str(FULL_DATA_YAML),
        "epochs": 30,
        "imgsz": 640,
        "batch": 16,
        "name": "yolo11n_full_training",
        "patience": 10,
    },
}

cfg = TRAINING_CONFIGS[TRAIN_MODE]
print("Training mode:", TRAIN_MODE)
print("Training config:", cfg)
print("Device:", DEVICE)

## 8. Train model

This cell trains either on the mini dataset or on the full dataset, depending on `TRAIN_MODE`.

In [ ]:
model = YOLO(MODEL_WEIGHTS)

train_results = model.train(
    data=cfg["data"],
    epochs=cfg["epochs"],
    imgsz=cfg["imgsz"],
    batch=cfg["batch"],
    project=PROJECT_NAME,
    name=cfg["name"],
    seed=SEED,
    patience=cfg["patience"],
    plots=True,
    device=DEVICE,
    exist_ok=True,
)

RUN_DIR = Path(PROJECT_NAME) / cfg["name"]
BEST_MODEL_PATH = RUN_DIR / "weights" / "best.pt"
LAST_MODEL_PATH = RUN_DIR / "weights" / "last.pt"

print("Run directory:", RUN_DIR)
print("Best model path:", BEST_MODEL_PATH)
print("Best model exists:", BEST_MODEL_PATH.exists())

## 9. Training curves / logged results

Ultralytics saves the training log as `results.csv`. This cell displays it as a table.

In [ ]:
results_csv = RUN_DIR / "results.csv"

if results_csv.exists():
    training_log = pd.read_csv(results_csv)
    display(training_log.tail())
else:
    print("No results.csv found yet:", results_csv)

## 10. Validation and test metrics

Important metrics for your report:

- Precision
- Recall
- mAP@0.5
- mAP@0.5:0.95

In [ ]:
def summarize_yolo_metrics(metrics, split_name):
    """Convert Ultralytics detection metrics into a small dataframe."""
    box = metrics.box
    summary = {
        "split": split_name,
        "precision_mean": float(box.mp),
        "recall_mean": float(box.mr),
        "mAP50": float(box.map50),
        "mAP50_95": float(box.map),
    }
    return pd.DataFrame([summary])


if not BEST_MODEL_PATH.exists():
    raise FileNotFoundError(f"Best model not found: {BEST_MODEL_PATH}. Run training first.")

best_model = YOLO(str(BEST_MODEL_PATH))

val_metrics = best_model.val(
    data=cfg["data"],
    split="val",
    imgsz=cfg["imgsz"],
    batch=cfg["batch"],
    project=PROJECT_NAME,
    name=f"{cfg['name']}_val_metrics",
    plots=True,
    device=DEVICE,
    exist_ok=True,
)

test_metrics = best_model.val(
    data=cfg["data"],
    split="test",
    imgsz=cfg["imgsz"],
    batch=cfg["batch"],
    project=PROJECT_NAME,
    name=f"{cfg['name']}_test_metrics",
    plots=True,
    device=DEVICE,
    exist_ok=True,
)

metrics_df = pd.concat([
    summarize_yolo_metrics(val_metrics, "val"),
    summarize_yolo_metrics(test_metrics, "test"),
], ignore_index=True)

metrics_output_path = RUN_DIR / "metrics_summary.csv"
metrics_df.to_csv(metrics_output_path, index=False)

print("Saved metrics summary to:", metrics_output_path)
display(metrics_df)

## 11. Run predictions on sample test images

This saves predicted images with bounding boxes, useful for the report.

In [ ]:
def load_yaml(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)


def resolve_dataset_split(data_yaml_path, split="test"):
    data = load_yaml(data_yaml_path)
    root = Path(data["path"])
    split_value = data.get(split) or data.get("val")
    return root / split_value


PREDICTION_SAMPLE_SIZE = 12
TEST_IMAGES_FOR_PREDICTION = resolve_dataset_split(cfg["data"], "test")

sample_images = list_image_files(TEST_IMAGES_FOR_PREDICTION)[:PREDICTION_SAMPLE_SIZE]
print("Prediction source folder:", TEST_IMAGES_FOR_PREDICTION)
print("Number of sample images:", len(sample_images))
print("First sample:", sample_images[0] if sample_images else "No images found")

prediction_results = best_model.predict(
    source=[str(p) for p in sample_images],
    imgsz=cfg["imgsz"],
    conf=0.25,
    save=True,
    project=PROJECT_NAME,
    name=f"{cfg['name']}_predictions",
    device=DEVICE,
    exist_ok=True,
)

PREDICTIONS_DIR = Path(PROJECT_NAME) / f"{cfg['name']}_predictions"
print("Predictions saved to:", PREDICTIONS_DIR)

In [ ]:
# Display a few saved prediction images
prediction_images = list_image_files(PREDICTIONS_DIR)
print("Saved prediction images:", len(prediction_images))

for img_path in prediction_images[:6]:
    print(img_path.name)
    display(Image(filename=str(img_path), width=500))

## 12. Optional: predict on your own image

Set `MY_IMAGE_PATH` to a local image path and run the cell.

In [ ]:
MY_IMAGE_PATH = None  # example: "my_car_photo.jpg"

if MY_IMAGE_PATH is not None:
    my_results = best_model.predict(
        source=MY_IMAGE_PATH,
        imgsz=cfg["imgsz"],
        conf=0.25,
        save=True,
        project=PROJECT_NAME,
        name=f"{cfg['name']}_my_image",
        device=DEVICE,
        exist_ok=True,
    )
    print("Done. Check saved prediction folder.")
else:
    print("Set MY_IMAGE_PATH to run prediction on your own image.")

## Notes for the report

This notebook currently trains and evaluates the **license plate detection** part:

```text
image → YOLO → bounding box around license plate
```

For the full project topic, the next step is OCR:

```text
image → YOLO detection → crop license plate → OCR → recognized plate text
```

The metrics produced here are detection metrics, not OCR text-recognition metrics.